# Estacionamento Inteligente: Validadores & Scanner

Guilherme Pedigone RA: 2712633
Laryssa Dantas RA: 2342968
Gabriel Diogo RA: 2683210

Projeto da Prática 3 de Compiladores (Aula CP 08 - Autômatos Finitos na Prática).

**Tema:** estacionamento inteligente. Fabricamos os autômatos na mão, em Python puro:
2 validadores AFD (placa Mercosul/antiga no mesmo autômato e horário `HH:MM`),
1 máquina de estados de negócio (a cancela) e 1 mini-scanner da linguagem de comandos do estacionamento.

| Célula | O que tem |
|---|---|
| 0 | Preparação do ambiente |
| 1 | Classe `AFD` (motor reutilizável, `rodar()`, `tabela()` e `minimizar()`) |
| 2 | Validador 1: placa (Mercosul `ABC1D23` e antiga `ABC-1234` no mesmo AFD) |
| 3 | Validador 2: horário `HH:MM` |
| 4 | Bateria de testes AFD x RegEx |
| 5 | Máquina de estados: a cancela |
| 6 | Scanner (função) |
| 7 | Scanner: programa correto, tabela de tokens e Maximal Munch |
| 8 | Scanner: erros léxicos com linha, coluna e dica |
| 9 | Bônus: AFD mínimo |
| 10 | Laboratório interativo (painel ipywidgets) |
| 11 | Resumo final |

## Célula 0 - Preparação do ambiente

In [1]:
# CÉLULA 0 — Preparação do ambiente
# Só usamos a biblioteca padrão do Python. A única coisa extra é o ipywidgets (Célula 10).
import re
print("✅ Ambiente pronto! Vamos construir os autômatos do estacionamento.")

✅ Ambiente pronto! Vamos construir os autômatos do estacionamento.


## Célula 1 - A classe `AFD`

O motor é sempre o mesmo: começar em q0, seguir as setas e conferir se terminou em F. O que muda de um problema para outro é só a tabela `delta`.
Além de `rodar()` e `tabela()`, a classe tem `minimizar()` (usado no bônus da Célula 9).

In [2]:
# CÉLULA 1 — A classe AFD reutilizável (com rastreamento passo a passo)
# Um AFD é a 5-upla M = (Q, Σ, δ, q0, F).
class AFD:
    def __init__(self, nome, delta, inicial, finais, classificar):
        self.nome = nome
        self.delta = delta                    # δ: {(estado, classe): próximo}
        self.inicial = inicial                # q0
        self.finais = set(finais)             # F
        self.classificar = classificar        # função: caractere -> classe
        # Q e Σ são calculados a partir do delta (na ordem em que aparecem)
        ordem = [inicial]
        for (e, _), p in delta.items():
            ordem += [e, p]
        self.estados = list(dict.fromkeys(ordem))                 # Q
        self.alfabeto = list(dict.fromkeys(s for _, s in delta))  # Σ (as classes usadas)

    def rodar(self, palavra, mostrar=True):
        """Lê a palavra caractere a caractere. Retorna True se aceita."""
        estado = self.inicial
        if mostrar:
            print(f"🔎 {self.nome} lendo {palavra!r}")
            print(f"  {'passo':5} {'lido':6} {'classe':7} {'de':6} para")
        for passo, c in enumerate(palavra, start=1):
            classe = self.classificar(c)
            proximo = self.delta.get((estado, classe))
            if mostrar:
                print(f"  {passo:<5} {c!r:6} {classe:7} {estado:6} {proximo or '☠ morto'}")
            if proximo is None:
                if mostrar:
                    print(f"  ❌ REJEITADA: morreu no meio, não há transição de {estado} com a classe {classe!r}\n")
                return False
            estado = proximo
        aceita = estado in self.finais
        if mostrar:
            if aceita:
                print(f"  ✅ ACEITA: terminou em {estado}, que é de aceitação\n")
            else:
                print(f"  ❌ REJEITADA: terminou em {estado}, que NÃO é de aceitação (faltou texto)\n")
        return aceita

    def tabela(self):
        """Imprime a tabela de transição: linhas = estados, colunas = classes."""
        print(f"📋 Tabela de transição — {self.nome}")
        print("     " + f"{'estado':8}" + "".join(f"{s:>6}" for s in self.alfabeto))
        for q in self.estados:
            marca = ("→" if q == self.inicial else " ") + ("*" if q in self.finais else " ")
            linha = "".join(f"{self.delta.get((q, s), '—'):>6}" for s in self.alfabeto)
            print(f"  {marca} {q:8}{linha}")
        print("  (→ = inicial, * = aceitação, — = vai para o estado morto)\n")

    def minimizar(self):
        """Algoritmo de partições (Moore). Retorna a lista de grupos de estados equivalentes."""
        MORTO = "☠"
        estados = self.estados + [MORTO]
        def d(q, s):
            return MORTO if q == MORTO else self.delta.get((q, s), MORTO)
        bloco = {q: (q in self.finais) for q in estados}          # partição inicial: F x não-F
        while True:
            assinatura = {q: (bloco[q], tuple(bloco[d(q, s)] for s in self.alfabeto)) for q in estados}
            ids = {}
            novo = {q: ids.setdefault(assinatura[q], len(ids)) for q in estados}
            estavel = len(set(novo.values())) == len(set(bloco.values()))
            bloco = novo
            if estavel:
                break
        grupos = {}
        for q in estados:
            grupos.setdefault(bloco[q], []).append(q)
        return [g for g in grupos.values() if MORTO not in g]

print("✅ Classe AFD definida.")

✅ Classe AFD definida.


## Célula 2 - Validador 1: placa (Mercosul e antiga no mesmo AFD)

A placa Mercosul tem o formato Letra Letra Letra Dígito Letra Dígito Dígito (`BRA2E19`) e a antiga tem Letra Letra Letra, hífen e 4 dígitos (`ABC-1234`).
Em vez de desenhar uma seta para cada uma das 26 letras e cada um dos 10 dígitos, agrupamos os caracteres em classes: `L` (letra maiúscula), `D` (dígito) e `-`.

Os 3 primeiros estados são iguais nos dois formatos. Depois da 3ª letra o autômato se divide: um dígito segue o caminho Mercosul, um hífen segue o caminho antigo. Os dois caminhos voltam a se encontrar nos estados `q5` e `q6`, porque ali falta a mesma coisa (2 dígitos, depois 1 dígito).

```
→ (q0) ─L→ (q1) ─L→ (q2) ─L→ (q3) ─D→ (q4) ─L→ (q5) ─D→ (q6) ─D→ ((qF))
                              │                  ▲
                              └─ - ─→ (qH) ─D→ (qA) ─D─┘          (antiga: ABC-1234)
```

In [3]:
# CÉLULA 2 — Validador 1: placa Mercosul (ABC1D23) e antiga (ABC-1234) no mesmo AFD
def classe_placa(c):
    if "A" <= c <= "Z": return "L"        # letra maiúscula
    if "0" <= c <= "9": return "D"        # dígito
    if c == "-":        return "-"
    return "outro"

delta_placa = {
    ("q0", "L"): "q1",
    ("q1", "L"): "q2",
    ("q2", "L"): "q3",
    ("q3", "D"): "q4",    # Mercosul: dígito depois das 3 letras
    ("q3", "-"): "qH",    # antiga: hífen depois das 3 letras
    ("q4", "L"): "q5",    # Mercosul: a letra do meio (ABC1D..)
    ("qH", "D"): "qA",    # antiga: 1º dígito
    ("qA", "D"): "q5",    # antiga: 2º dígito (daqui faltam 2 dígitos, igual ao Mercosul)
    ("q5", "D"): "q6",
    ("q6", "D"): "qF",
}
placa = AFD("Placa (Mercosul e antiga)", delta_placa, "q0", {"qF"}, classe_placa)
placa.tabela()
placa.rodar("BRA2E19")     # aceita (Mercosul)
placa.rodar("ABC-1234")    # aceita (antiga)
placa.rodar("ABC1234")     # rejeitada: morre no meio (faltou a letra do meio)
placa.rodar("ABC1D2")      # rejeitada: termina fora de F (faltou um dígito)

📋 Tabela de transição — Placa (Mercosul e antiga)
     estado       L     D     -
  →  q0          q1     —     —
     q1          q2     —     —
     q2          q3     —     —
     q3           —    q4    qH
     q4          q5     —     —
     qH           —    qA     —
     q5           —    q6     —
     qA           —    q5     —
     q6           —    qF     —
   * qF           —     —     —
  (→ = inicial, * = aceitação, — = vai para o estado morto)

🔎 Placa (Mercosul e antiga) lendo 'BRA2E19'
  passo lido   classe  de     para
  1     'B'    L       q0     q1
  2     'R'    L       q1     q2
  3     'A'    L       q2     q3
  4     '2'    D       q3     q4
  5     'E'    L       q4     q5
  6     '1'    D       q5     q6
  7     '9'    D       q6     qF
  ✅ ACEITA: terminou em qF, que é de aceitação

🔎 Placa (Mercosul e antiga) lendo 'ABC-1234'
  passo lido   classe  de     para
  1     'A'    L       q0     q1
  2     'B'    L       q1     q2
  3     'C'    L       q2     q3


False

## Célula 3 - Validador 2: horário `HH:MM` (00:00 até 23:59)

Aqui a classe "dígito" não basta, então os dígitos são divididos em classes finas: `0-1`, `2`, `3`, `4-5` e `6-9`.
Se a hora começa com 0 ou 1, o segundo dígito é livre; se começa com 2, só vai até 3; e o primeiro dígito do minuto só vai até 5.

```
                    0-1,2,3,4-5,6-9
→ (H0) ──0-1──→ (H1) ──────────────┐
    │                              ▼
    └───2────→ (H2) ──0-1,2,3──→ (SEP) ──:──→ (M0) ──0-1,2,3,4-5──→ (M1) ──0-1,2,3,4-5,6-9──→ ((FIM))
```

In [4]:
# CÉLULA 3 — Validador 2: horário HH:MM (00:00 a 23:59)
def classe_hora(c):
    if c in "01":   return "0-1"
    if c == "2":    return "2"
    if c == "3":    return "3"
    if c in "45":   return "4-5"
    if c in "6789": return "6-9"
    if c == ":":    return ":"
    return "outro"

DIGITOS = ["0-1", "2", "3", "4-5", "6-9"]
delta_hora = {("H0", "0-1"): "H1", ("H0", "2"): "H2"}
for d in DIGITOS:                              # hora 0x ou 1x: segundo dígito livre
    delta_hora[("H1", d)] = "SEP"
for d in ["0-1", "2", "3"]:                    # hora 2x: segundo dígito só até 3
    delta_hora[("H2", d)] = "SEP"
delta_hora[("SEP", ":")] = "M0"
for d in ["0-1", "2", "3", "4-5"]:             # minuto: dezena só até 5
    delta_hora[("M0", d)] = "M1"
for d in DIGITOS:                              # minuto: unidade livre
    delta_hora[("M1", d)] = "FIM"

hora = AFD("Horário HH:MM", delta_hora, "H0", {"FIM"}, classe_hora)
hora.tabela()
hora.rodar("08:15")     # aceita
hora.rodar("25:10")     # rejeitada: morre no meio (hora 25)
hora.rodar("10:4")      # rejeitada: termina fora de F (faltou um dígito)

📋 Tabela de transição — Horário HH:MM
     estado     0-1     2     3   4-5   6-9     :
  →  H0          H1    H2     —     —     —     —
     H1         SEP   SEP   SEP   SEP   SEP     —
     H2         SEP   SEP   SEP     —     —     —
     SEP          —     —     —     —     —    M0
     M0          M1    M1    M1    M1     —     —
     M1         FIM   FIM   FIM   FIM   FIM     —
   * FIM          —     —     —     —     —     —
  (→ = inicial, * = aceitação, — = vai para o estado morto)

🔎 Horário HH:MM lendo '08:15'
  passo lido   classe  de     para
  1     '0'    0-1     H0     H1
  2     '8'    6-9     H1     SEP
  3     ':'    :       SEP    M0
  4     '1'    0-1     M0     M1
  5     '5'    4-5     M1     FIM
  ✅ ACEITA: terminou em FIM, que é de aceitação

🔎 Horário HH:MM lendo '25:10'
  passo lido   classe  de     para
  1     '2'    2       H0     H2
  2     '5'    4-5     H2     ☠ morto
  ❌ REJEITADA: morreu no meio, não há transição de H2 com a classe '4-5'

🔎 Horário 

False

## Célula 4 - Bateria de testes AFD x RegEx

Para cada AFD existe uma RegEx equivalente, usada com `re.fullmatch` (o `fullmatch` evita aceitar texto com lixo antes ou depois).
Cada bateria tem pelo menos 5 palavras aceitas e 5 rejeitadas, incluindo a palavra vazia, e o esperado é **0 divergências**.
Depois, uma checagem exaustiva compara os dois em milhares de palavras geradas.

In [5]:
# CÉLULA 4 — Bateria de testes: AFD x RegEx
import itertools

REGEX_PLACA = re.compile(r"[A-Z]{3}([0-9][A-Z][0-9]{2}|-[0-9]{4})")
REGEX_HORA  = re.compile(r"([01][0-9]|2[0-3]):[0-5][0-9]")

TESTES = {
    "Placa": (placa, REGEX_PLACA,
        ["ABC1D23", "BRA2E19", "RIO2A34", "ABC-1234", "XYZ-0000", "QWE9Z99"],                              # aceitas
        ["", "ABC1234", "AB1C234", "abc1d23", "ABC1D2", "ABC-123", "ABC-12345", "ABC1D234", "ABC 1D23"]),   # rejeitadas
    "Horário HH:MM": (hora, REGEX_HORA,
        ["00:00", "08:15", "12:30", "18:30", "23:59", "19:45"],
        ["", "24:00", "18:75", "8:15", "18:3", "18-30", "12:60", "ab:cd", "23:599"]),
}

total_div = 0
for nome, (afd, regex, aceitas, rejeitadas) in TESTES.items():
    print(f"=== {nome} ===")
    div = 0
    for palavra, esperado in [(p, True) for p in aceitas] + [(p, False) for p in rejeitadas]:
        r_afd = afd.rodar(palavra, mostrar=False)
        r_re  = regex.fullmatch(palavra) is not None
        ok = (r_afd == r_re == esperado)
        div += (r_afd != r_re)
        print(f"  {palavra!r:11} AFD={str(r_afd):5} RegEx={str(r_re):5} esperado={str(esperado):5} {'✅' if ok else '❌ DIVERGÊNCIA'}")
    total_div += div
    print(f"  -> {len(aceitas)} aceitas, {len(rejeitadas)} rejeitadas, divergências: {div}\n")

# Checagem exaustiva: todas as palavras de até 8 caracteres do alfabeto {A, 1, -, a} para a placa
# e de até 5 caracteres do alfabeto {0,1,2,3,6,9,:} para o horário.
def exaustivo(afd, regex, simbolos, tamanho_max):
    n, div = 0, 0
    for tam in range(tamanho_max + 1):
        for tupla in itertools.product(simbolos, repeat=tam):
            palavra = "".join(tupla)
            n += 1
            div += (afd.rodar(palavra, mostrar=False) != (regex.fullmatch(palavra) is not None))
    return n, div

n1, d1 = exaustivo(placa, REGEX_PLACA, "A1-a", 8)
n2, d2 = exaustivo(hora, REGEX_HORA, "0123689:", 5)
print(f"Exaustivo placa:   {n1} palavras testadas, {d1} divergências")
print(f"Exaustivo horário: {n2} palavras testadas, {d2} divergências")
assert total_div == 0 and d1 == 0 and d2 == 0
print("\n✅ AFD e RegEx concordam em todas as palavras.")

=== Placa ===
  'ABC1D23'   AFD=True  RegEx=True  esperado=True  ✅
  'BRA2E19'   AFD=True  RegEx=True  esperado=True  ✅
  'RIO2A34'   AFD=True  RegEx=True  esperado=True  ✅
  'ABC-1234'  AFD=True  RegEx=True  esperado=True  ✅
  'XYZ-0000'  AFD=True  RegEx=True  esperado=True  ✅
  'QWE9Z99'   AFD=True  RegEx=True  esperado=True  ✅
  ''          AFD=False RegEx=False esperado=False ✅
  'ABC1234'   AFD=False RegEx=False esperado=False ✅
  'AB1C234'   AFD=False RegEx=False esperado=False ✅
  'abc1d23'   AFD=False RegEx=False esperado=False ✅
  'ABC1D2'    AFD=False RegEx=False esperado=False ✅
  'ABC-123'   AFD=False RegEx=False esperado=False ✅
  'ABC-12345' AFD=False RegEx=False esperado=False ✅
  'ABC1D234'  AFD=False RegEx=False esperado=False ✅
  'ABC 1D23'  AFD=False RegEx=False esperado=False ✅
  -> 6 aceitas, 9 rejeitadas, divergências: 0

=== Horário HH:MM ===
  '00:00'     AFD=True  RegEx=True  esperado=True  ✅
  '08:15'     AFD=True  RegEx=True  esperado=True  ✅
  '12:30'     AF

## Célula 5 - Máquina de estados de negócio: a cancela

Usa a mesma ideia do AFD, só que os símbolos agora são **eventos** do estacionamento. São 5 estados e 9 transições.
O `sensor_obstaculo` faz a cancela reabrir quando ela está fechando, e a `falha_motor` trava a cancela até a manutenção.
Quando um evento é recusado, a máquina mostra quais eventos são permitidos naquele estado.

```
FECHADA ─carro_detectado→ ABRINDO ─abertura_completa→ ABERTA ─carro_passou→ FECHANDO ─fechamento_completo→ FECHADA

ABERTA   ─tempo_esgotado→ FECHANDO          FECHANDO ─sensor_obstaculo→ ABRINDO   (a cancela reabre)
ABRINDO  ─falha_motor→ TRAVADA              FECHANDO ─falha_motor→ TRAVADA        TRAVADA ─manutencao_ok→ FECHADA
```

In [6]:
# CÉLULA 5 — Máquina de estados de negócio: a cancela do estacionamento
class MaquinaEstados:
    def __init__(self, nome, transicoes, inicial, finais, acoes=None):
        self.nome, self.delta = nome, transicoes          # {(estado, evento): próximo}
        self.inicial, self.finais = inicial, set(finais)
        self.acoes = acoes or {}                          # ação executada ao ENTRAR num estado
        self.estados = list(dict.fromkeys([inicial] + [x for (e, _), p in transicoes.items() for x in (e, p)]))
        self.reiniciar()

    def reiniciar(self):
        self.estado = self.inicial

    def permitidos(self, estado=None):
        estado = estado or self.estado
        return [ev for (e, ev) in self.delta if e == estado]

    def enviar(self, evento, mostrar=True):
        prox = self.delta.get((self.estado, evento))
        if prox is None:
            if mostrar:
                print(f"  ❌ evento {evento!r} RECUSADO em {self.estado}. Permitidos: {', '.join(self.permitidos())}")
            return False
        if mostrar:
            extra = f"   [{self.acoes[prox]}]" if prox in self.acoes else ""
            print(f"  ✅ {self.estado:9} --{evento}--> {prox}{extra}")
        self.estado = prox
        return True

    def executar(self, eventos, mostrar=True):
        """Roda um fluxo inteiro. Retorna True se nenhum evento foi recusado e terminou em estado final."""
        self.reiniciar()
        if mostrar:
            print(f"▶ {self.nome}: {' → '.join(eventos)}")
        for ev in eventos:
            if not self.enviar(ev, mostrar):
                if mostrar: print("  ⛔ fluxo INVÁLIDO\n")
                return False
        ok = self.estado in self.finais
        if mostrar:
            print(f"  {'🏁 fluxo VÁLIDO' if ok else '⛔ fluxo INVÁLIDO'}: terminou em {self.estado}"
                  f"{'' if ok else ', que não é final'}\n")
        return ok

cancela = MaquinaEstados("Cancela", {
    ("FECHADA",   "carro_detectado"):    "ABRINDO",
    ("ABRINDO",   "abertura_completa"):  "ABERTA",
    ("ABERTA",    "carro_passou"):       "FECHANDO",
    ("ABERTA",    "tempo_esgotado"):     "FECHANDO",
    ("FECHANDO",  "fechamento_completo"):"FECHADA",
    ("FECHANDO",  "sensor_obstaculo"):   "ABRINDO",
    ("ABRINDO",   "falha_motor"):        "TRAVADA",
    ("FECHANDO",  "falha_motor"):        "TRAVADA",
    ("TRAVADA",   "manutencao_ok"):      "FECHADA",
}, inicial="FECHADA", finais={"FECHADA"}, acoes={
    "ABRINDO":  "abrir(cancela)",
    "FECHANDO": "fechar(cancela)",
    "TRAVADA":  "aviso(\"cancela travada\")",
})

print(f"{len(cancela.estados)} estados e {len(cancela.delta)} transições\n")
print("--- 2 fluxos VÁLIDOS ---")
cancela.executar(["carro_detectado", "abertura_completa", "carro_passou", "fechamento_completo"])
cancela.executar(["carro_detectado", "abertura_completa", "carro_passou", "sensor_obstaculo",
                  "abertura_completa", "carro_passou", "fechamento_completo"])
print("--- 2 fluxos INVÁLIDOS ---")
cancela.executar(["carro_detectado", "carro_passou"])                       # carro passou sem a cancela abrir
cancela.executar(["carro_detectado", "falha_motor", "abertura_completa"])   # tenta abrir com a cancela travada

5 estados e 9 transições

--- 2 fluxos VÁLIDOS ---
▶ Cancela: carro_detectado → abertura_completa → carro_passou → fechamento_completo
  ✅ FECHADA   --carro_detectado--> ABRINDO   [abrir(cancela)]
  ✅ ABRINDO   --abertura_completa--> ABERTA
  ✅ ABERTA    --carro_passou--> FECHANDO   [fechar(cancela)]
  ✅ FECHANDO  --fechamento_completo--> FECHADA
  🏁 fluxo VÁLIDO: terminou em FECHADA

▶ Cancela: carro_detectado → abertura_completa → carro_passou → sensor_obstaculo → abertura_completa → carro_passou → fechamento_completo
  ✅ FECHADA   --carro_detectado--> ABRINDO   [abrir(cancela)]
  ✅ ABRINDO   --abertura_completa--> ABERTA
  ✅ ABERTA    --carro_passou--> FECHANDO   [fechar(cancela)]
  ✅ FECHANDO  --sensor_obstaculo--> ABRINDO   [abrir(cancela)]
  ✅ ABRINDO   --abertura_completa--> ABERTA
  ✅ ABERTA    --carro_passou--> FECHANDO   [fechar(cancela)]
  ✅ FECHANDO  --fechamento_completo--> FECHADA
  🏁 fluxo VÁLIDO: terminou em FECHADA

--- 2 fluxos INVÁLIDOS ---
▶ Cancela: carro_detectado

False

## Célula 6 - Scanner da linguagem de comandos do estacionamento

O scanner lê os comandos do estacionamento, como `ENTRADA BRA2E19 08:15 VAGA A12` e `SAIDA BRA2E19 10:40 PAGO 12.00`, e entrega a lista de tokens.
Ele usa **Maximal Munch** (sempre tenta ler o maior lexema possível), ignora espaços e comentários (`# ...`) e, quando acha um erro, **registra e continua lendo**.

**Tipos de token (14):** `RESERVADA`, `ID`, `PLACA`, `HORARIO`, `CODIGO_VAGA`, `NUM_INT`, `NUM_REAL`, `STRING`, `MENOR`, `MENOR_IGUAL`, `MAIOR`, `MAIOR_IGUAL`, `IGUAL`, `DIFERENTE`.

**Palavras reservadas (9):** `ENTRADA`, `SAIDA`, `VAGA`, `PAGO`, `ABRIR`, `FECHAR`, `AVISO`, `SE`, `ENTAO`.

**Casos de Maximal Munch:** `ABC-1234` é uma `PLACA` inteira (e não `ID`, hífen e número), `A12` é `CODIGO_VAGA` (e não `ID`), `<` / `<=`, `>` / `>=`, `==`, e `18` / `12.00` / `08:15` (inteiro, real e horário).

Quem decide se um trecho é uma placa, um horário ou um código de vaga são os próprios AFDs: o scanner junta o trecho e pergunta ao autômato (Células 2 e 3, mais um AFD pequeno para o código da vaga).

In [7]:
# CÉLULA 6 — Scanner da linguagem do estacionamento
RESERVADAS = {"ENTRADA", "SAIDA", "VAGA", "PAGO", "ABRIR", "FECHAR", "AVISO", "SE", "ENTAO"}

# AFD pequeno só para o código da vaga: uma letra e 1 ou 2 dígitos (A1, B7, C12)
delta_vaga = {("v0", "L"): "v1", ("v1", "D"): "v2", ("v2", "D"): "v3"}
vaga = AFD("Código da vaga", delta_vaga, "v0", {"v2", "v3"}, classe_placa)

def eh_letra(c):
    return (c.isascii() and c.isalpha()) or c == "_"

def scanner(fonte):
    """Retorna (tokens, erros). Cada token é (tipo, lexema, linha, coluna)."""
    tokens, erros = [], []
    i, linha, coluna, n = 0, 1, 1, len(fonte)

    def erro(l, col, lexema, dica):
        erros.append(f"Linha {l}, coluna {col}: trecho inesperado {lexema!r} ({dica})")

    while i < n:
        c = fonte[i]
        if c == "\n":                                   # quebra de linha: só atualiza a posição
            i, linha, coluna = i + 1, linha + 1, 1
            continue
        if c in " \t\r":                                # espaço: ignorado
            i, coluna = i + 1, coluna + 1
            continue
        if c == "#":                                    # comentário: ignora até o fim da linha
            while i < n and fonte[i] != "\n":
                i, coluna = i + 1, coluna + 1
            continue

        inicio, col_inicio = i, coluna

        if eh_letra(c):                                 # palavra: reservada, placa, vaga ou identificador
            while i < n and (eh_letra(fonte[i]) or fonte[i].isdigit()):
                i += 1
            # Maximal Munch: placa antiga (ABC-1234) inclui o hífen e os dígitos
            if i + 1 < n and fonte[i] == "-" and fonte[i + 1].isdigit() and i - inicio == 3:
                i += 1
                while i < n and fonte[i].isdigit():
                    i += 1
            lexema = fonte[inicio:i]
            if placa.rodar(lexema, mostrar=False):
                tipo = "PLACA"
            elif vaga.rodar(lexema, mostrar=False):
                tipo = "CODIGO_VAGA"
            elif lexema in RESERVADAS:
                tipo = "RESERVADA"
            elif len(lexema) >= 4 and lexema[:3].isalpha() and lexema[3].isdigit() or "-" in lexema:
                erro(linha, col_inicio, lexema,
                     "placa inválida: use ABC1D23 (Mercosul) ou ABC-1234 (antiga), com letras maiúsculas")
                coluna += i - inicio
                continue
            else:
                tipo = "ID"

        elif c.isdigit():                               # número inteiro, real ou horário
            while i < n and fonte[i].isdigit():
                i += 1
            if i + 1 < n and fonte[i] == ":" and fonte[i + 1].isdigit():      # candidato a HH:MM
                i += 1
                while i < n and fonte[i].isdigit():
                    i += 1
                lexema = fonte[inicio:i]
                if hora.rodar(lexema, mostrar=False):                          # pergunta ao AFD da Célula 3
                    tipo = "HORARIO"
                else:
                    erro(linha, col_inicio, lexema, "horário vai de 00:00 a 23:59, com dois dígitos em cada parte")
                    coluna += i - inicio
                    continue
            elif i + 1 < n and fonte[i] == "." and fonte[i + 1].isdigit():    # real: 12.00
                i += 1
                while i < n and fonte[i].isdigit():
                    i += 1
                tipo = "NUM_REAL"
            elif i + 1 < n and fonte[i] == "," and fonte[i + 1].isdigit():    # 12,50 (vírgula decimal)
                i += 1
                while i < n and fonte[i].isdigit():
                    i += 1
                erro(linha, col_inicio, fonte[inicio:i], "use ponto como separador decimal: 12.50")
                coluna += i - inicio
                continue
            else:
                tipo = "NUM_INT"
            lexema = fonte[inicio:i]

        elif c in "<>":                                 # Maximal Munch: < / <=  e  > / >=
            if i + 1 < n and fonte[i + 1] == "=":
                i += 2
                tipo = "MENOR_IGUAL" if c == "<" else "MAIOR_IGUAL"
            else:
                i += 1
                tipo = "MENOR" if c == "<" else "MAIOR"
            lexema = fonte[inicio:i]

        elif c == "=":                                  # só existe ==
            if i + 1 < n and fonte[i + 1] == "=":
                i += 2
                tipo, lexema = "IGUAL", "=="
            else:
                erro(linha, col_inicio, "=", "para comparar use ==, por exemplo: SE livres == 0 ENTAO ...")
                i, coluna = i + 1, coluna + 1
                continue

        elif c == "!":                                  # só existe !=
            if i + 1 < n and fonte[i + 1] == "=":
                i += 2
                tipo, lexema = "DIFERENTE", "!="
            else:
                erro(linha, col_inicio, "!", "para diferente use !=")
                i, coluna = i + 1, coluna + 1
                continue

        elif c == '"':                                  # texto entre aspas, na mesma linha
            i += 1
            while i < n and fonte[i] not in '"\n':
                i += 1
            if i < n and fonte[i] == '"':
                i += 1
                tipo, lexema = "STRING", fonte[inicio:i]
            else:
                erro(linha, col_inicio, fonte[inicio:i], 'texto sem aspas de fechamento: AVISO "pátio lotado"')
                coluna += i - inicio
                continue

        else:                                           # nenhum estado aceita: erro léxico
            if c == "$":
                dica = "não escreva a moeda: o valor é só o número, por exemplo PAGO 12.50"
            elif c == "-":
                dica = "hífen só aparece dentro da placa antiga (ABC-1234); valores não são negativos"
            elif c == ".":
                dica = "o ponto decimal precisa de dígito antes e depois: 12.50"
            else:
                dica = "caractere fora da linguagem"
            erro(linha, col_inicio, c, dica)
            i, coluna = i + 1, coluna + 1
            continue

        tokens.append((tipo, lexema, linha, col_inicio))
        coluna += i - inicio

    return tokens, erros

def mostrar_tokens(tokens):
    print(f"{'TIPO':12} {'LEXEMA':22} {'LIN':>3} {'COL':>3}")
    for tipo, lexema, lin, col in tokens:
        print(f"{tipo:12} {lexema:22} {lin:>3} {col:>3}")

print("✅ Scanner definido.")

✅ Scanner definido.


## Célula 7 - Scanner: programa correto, tabela de tokens e Maximal Munch

In [8]:
# CÉLULA 7 — Scanner lendo um movimento correto do estacionamento
PROGRAMA_OK = """# movimento do estacionamento
ENTRADA BRA2E19 08:15 VAGA A12
ENTRADA ABC-1234 08:40 VAGA B7
SAIDA BRA2E19 10:40 PAGO 12.00
SE livres <= 5 ENTAO AVISO "patio quase lotado"
ABRIR cancela
"""
tokens, erros = scanner(PROGRAMA_OK)
mostrar_tokens(tokens)
print(f"\n{len(tokens)} tokens, {len(erros)} erros")
print("Tipos distintos encontrados:", len({t[0] for t in tokens}))
assert not erros

# Maximal Munch: o scanner sempre escolhe o MAIOR lexema possível
print("\n--- Maximal Munch ---")
for texto in ["ABC-1234 BRA2E19", "A12 B7 ABC", "< <= > >= == !=", "18 12.00 08:15"]:
    toks, errs = scanner(texto)
    print(f"{texto!r:22} -> {[(t[0], t[1]) for t in toks]}")

TIPO         LEXEMA                 LIN COL
RESERVADA    ENTRADA                  2   1
PLACA        BRA2E19                  2   9
HORARIO      08:15                    2  17
RESERVADA    VAGA                     2  23
CODIGO_VAGA  A12                      2  28
RESERVADA    ENTRADA                  3   1
PLACA        ABC-1234                 3   9
HORARIO      08:40                    3  18
RESERVADA    VAGA                     3  24
CODIGO_VAGA  B7                       3  29
RESERVADA    SAIDA                    4   1
PLACA        BRA2E19                  4   7
HORARIO      10:40                    4  15
RESERVADA    PAGO                     4  21
NUM_REAL     12.00                    4  26
RESERVADA    SE                       5   1
ID           livres                   5   4
MENOR_IGUAL  <=                       5  11
NUM_INT      5                        5  14
RESERVADA    ENTAO                    5  16
RESERVADA    AVISO                    5  22
STRING       "patio quase lotado

## Célula 8 - Scanner: erros léxicos com linha, coluna e dica (e continuando depois do erro)

In [9]:
# CÉLULA 8 — Scanner com erros: mostra todos e continua lendo
PROGRAMA_ERROS = """ENTRADA ABC1234 08:15 VAGA A12
SAIDA BRA2E19 25:10 PAGO R$12,50
ENTRADA XYZ-12 09:00 VAGA C3
SE livres = 5 ENTAO AVISO "patio lotado
FECHAR cancela
"""
tokens, erros = scanner(PROGRAMA_ERROS)
print(f"{len(erros)} erros léxicos encontrados:\n")
for e in erros:
    print(" ❌", e)
print(f"\nO scanner continuou lendo e ainda produziu {len(tokens)} tokens válidos.")
print("Último token lido:", tokens[-1])
assert len(erros) == 7 and tokens[-1][1] == "cancela"

7 erros léxicos encontrados:

 ❌ Linha 1, coluna 9: trecho inesperado 'ABC1234' (placa inválida: use ABC1D23 (Mercosul) ou ABC-1234 (antiga), com letras maiúsculas)
 ❌ Linha 2, coluna 15: trecho inesperado '25:10' (horário vai de 00:00 a 23:59, com dois dígitos em cada parte)
 ❌ Linha 2, coluna 27: trecho inesperado '$' (não escreva a moeda: o valor é só o número, por exemplo PAGO 12.50)
 ❌ Linha 2, coluna 28: trecho inesperado '12,50' (use ponto como separador decimal: 12.50)
 ❌ Linha 3, coluna 9: trecho inesperado 'XYZ-12' (placa inválida: use ABC1D23 (Mercosul) ou ABC-1234 (antiga), com letras maiúsculas)
 ❌ Linha 4, coluna 11: trecho inesperado '=' (para comparar use ==, por exemplo: SE livres == 0 ENTAO ...)
 ❌ Linha 4, coluna 27: trecho inesperado '"patio lotado' (texto sem aspas de fechamento: AVISO "pátio lotado")

O scanner continuou lendo e ainda produziu 19 tokens válidos.
Último token lido: ('ID', 'cancela', 5, 8)


## Célula 9 - Bônus: AFD mínimo

Rodamos o algoritmo de partições (Moore, da Aula 07) nos dois validadores. Se o número de grupos for igual ao número de estados, **nenhum par de estados é equivalente** e o AFD já é mínimo.

A placa foi desenhada primeiro com um caminho separado para a placa antiga até o fim (13 estados). O algoritmo mostra que os estados do fim dos dois caminhos fazem a mesma coisa e podem ser fundidos, o que dá os 10 estados do AFD final.

In [10]:
# CÉLULA 9 — Bônus: mostrando que os AFDs finais são mínimos
for afd in (placa, hora):
    grupos = afd.minimizar()
    print(f"{afd.nome}: {len(afd.estados)} estados -> {len(grupos)} grupos de equivalência")
    for g in grupos:
        print("   ", g)
    print("   ✅ AFD já é mínimo (nenhum par de estados equivalentes)\n" if len(grupos) == len(afd.estados)
          else "   ⚠ há estados equivalentes, dá para minimizar\n")

# O primeiro rascunho da placa: o caminho da placa antiga tinha estados próprios até o fim
delta_rascunho = dict(delta_placa)
for chave in [("qA", "D"), ("q5", "D"), ("q6", "D")]:
    del delta_rascunho[chave]
delta_rascunho.update({
    ("q5", "D"): "q6", ("q6", "D"): "q7",           # Mercosul termina em q7
    ("qA", "D"): "a2", ("a2", "D"): "a3", ("a3", "D"): "a4",   # antiga: a2, a3 e a4 repetem q5, q6 e q7
})
rascunho = AFD("Rascunho da placa", delta_rascunho, "q0", {"q7", "a4"}, classe_placa)
grupos = rascunho.minimizar()
print(f"{rascunho.nome}: {len(rascunho.estados)} estados -> {len(grupos)} grupos")
for g in grupos:
    if len(g) > 1:
        print("    estados equivalentes, podem ser fundidos:", g)
assert len(rascunho.estados) == 13 and len(grupos) == 10

Placa (Mercosul e antiga): 10 estados -> 10 grupos de equivalência
    ['q0']
    ['q1']
    ['q2']
    ['q3']
    ['q4']
    ['qH']
    ['q5']
    ['qA']
    ['q6']
    ['qF']
   ✅ AFD já é mínimo (nenhum par de estados equivalentes)

Horário HH:MM: 7 estados -> 7 grupos de equivalência
    ['H0']
    ['H1']
    ['H2']
    ['SEP']
    ['M0']
    ['M1']
    ['FIM']
   ✅ AFD já é mínimo (nenhum par de estados equivalentes)

Rascunho da placa: 13 estados -> 10 grupos
    estados equivalentes, podem ser fundidos: ['q5', 'a2']
    estados equivalentes, podem ser fundidos: ['q6', 'a3']
    estados equivalentes, podem ser fundidos: ['q7', 'a4']


## Célula 10 - Laboratório interativo

Painel com `ipywidgets`: escolha o validador, a máquina de estados ou o scanner, digite o texto e clique em **Rodar**.
No VS Code, instale antes `ipywidgets` (`pip install ipywidgets`).

In [11]:
# CÉLULA 10 — Laboratório interativo (painel ipywidgets)
import ipywidgets as widgets
from IPython.display import display

EXEMPLOS = {
    "AFD — Placa":              "BRA2E19",
    "AFD — Horário HH:MM":      "08:15",
    "Máquina — Cancela":        "carro_detectado abertura_completa carro_passou sensor_obstaculo abertura_completa carro_passou fechamento_completo",
    "Scanner — Estacionamento": 'ENTRADA BRA2E19 08:15 VAGA A12\nSAIDA BRA2E19 10:40 PAGO 12.00',
}

def executar_painel(opcao, texto):
    """Lógica do painel (separada dos widgets para poder ser testada)."""
    if opcao == "AFD — Placa":
        placa.rodar(texto)
    elif opcao == "AFD — Horário HH:MM":
        hora.rodar(texto)
    elif opcao == "Máquina — Cancela":
        cancela.executar(texto.replace(",", " ").split())
    else:
        toks, errs = scanner(texto)
        mostrar_tokens(toks)
        for e in errs:
            print(" ❌", e)
        print(f"\n{len(toks)} tokens, {len(errs)} erros")

escolha = widgets.Dropdown(options=list(EXEMPLOS), description="Teste:", layout=widgets.Layout(width="320px"))
entrada = widgets.Textarea(value=EXEMPLOS[escolha.value], description="Texto:",
                           layout=widgets.Layout(width="520px", height="110px"))
botao   = widgets.Button(description="Rodar", button_style="primary")
saida   = widgets.Output()

def ao_trocar(mudanca):
    entrada.value = EXEMPLOS[mudanca["new"]]          # sugere um exemplo para cada opção
escolha.observe(ao_trocar, names="value")

def ao_clicar(_):
    saida.clear_output()
    with saida:
        executar_painel(escolha.value, entrada.value)
botao.on_click(ao_clicar)

display(widgets.VBox([escolha, entrada, botao]), saida)

## Célula 11 - Resumo final

In [12]:
# CÉLULA 11 — Resumo: confere os requisitos da Prática 3
checks = {
    "2 validadores AFD com >= 5 estados":          len(placa.estados) >= 5 and len(hora.estados) >= 5,
    "classes finas no AFD do horário (>= 5)":      len(hora.alfabeto) >= 5,
    "AFD x RegEx sem divergências":                total_div == 0 and d1 == 0 and d2 == 0,
    "máquina com >= 5 estados e >= 6 transições":  len(cancela.estados) >= 5 and len(cancela.delta) >= 6,
    "2 fluxos válidos e 2 inválidos":              [cancela.executar(f, False) for f in (
        ["carro_detectado", "abertura_completa", "carro_passou", "fechamento_completo"],
        ["carro_detectado", "abertura_completa", "carro_passou", "sensor_obstaculo", "abertura_completa", "carro_passou", "fechamento_completo"],
        ["carro_detectado", "carro_passou"],
        ["carro_detectado", "falha_motor", "abertura_completa"])] == [True, True, False, False],
    "scanner com >= 8 tipos de token":             len({t[0] for t in scanner(PROGRAMA_OK)[0]}) >= 8,
    "scanner com >= 3 palavras reservadas":        len(RESERVADAS) >= 3,
    "scanner continua depois do erro":             scanner(PROGRAMA_ERROS)[0][-1][1] == "cancela",
}
for nome, ok in checks.items():
    print(("✅" if ok else "❌"), nome)
assert all(checks.values())
print("\n🎉 Tudo certo! Pronto para entregar.")

✅ 2 validadores AFD com >= 5 estados
✅ classes finas no AFD do horário (>= 5)
✅ AFD x RegEx sem divergências
✅ máquina com >= 5 estados e >= 6 transições
✅ 2 fluxos válidos e 2 inválidos
✅ scanner com >= 8 tipos de token
✅ scanner com >= 3 palavras reservadas
✅ scanner continua depois do erro

🎉 Tudo certo! Pronto para entregar.
